# 02 — Text Cleaning

Goal: take the raw text column and produce a cleaned version that's ready for the sentiment model.

What we'll do and why:
- **Remove diacritics** — those small vowel markers above/below Arabic letters (ِ ً ُ etc). Models don't need them and they create inconsistency (same word written with and without diacritics = two different tokens).
- **Normalize letter forms** — Arabic has alternate forms of some letters (e.g. ة vs ه at end of words, أ/إ/آ vs ا). Normalize them so the model sees consistent vocabulary.
- **Remove noise** — URLs, @mentions, extra whitespace, non-Arabic characters that add no signal.
- **Inspect the outlier** — that 864-char tweet from EDA. Decide whether to keep or drop.
- **Save** the cleaned DataFrame to `data/processed/` so we don't redo this step every time.

In [ ]:
import pandas as pd
import numpy as np
import re
import sys
sys.path.append('..')

df_raw = pd.read_csv('../data/raw/ajgt_twitter_ar.csv')
df_raw['sentiment'] = df_raw['label'].map({1: 'positive', 0: 'negative'})

print(f'Loaded {len(df_raw)} rows')
df_raw.head(5)

## Step 1 — Inspect the outlier

EDA showed one tweet at 864 characters. Before we clean anything, let's read it.
If it's a genuine post, keep it. If it's corrupted data or a scraping artifact, drop it.

In [ ]:
df_raw['char_count'] = df_raw['text'].str.len()

# Show the longest 3 tweets
pd.set_option('display.max_colwidth', 1000)
df_raw.nlargest(3, 'char_count')[['text', 'sentiment', 'char_count']]

## Step 2 — Build the cleaning function

This is the core of the notebook. Each operation is a separate line so you can see exactly what's happening.
The order matters — for example, you want to remove diacritics *before* you normalize letter forms.

In [ ]:
def clean_arabic(text: str) -> str:
    """
    Clean a single Arabic tweet.
    Each step is explicit so it's easy to debug or modify.
    """
    if not isinstance(text, str) or not text.strip():
        return ""

    # 1. Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # 2. Remove @mentions
    text = re.sub(r'@\w+', '', text)

    # 3. Remove hashtag symbol but keep the word  (#تونس → تونس)
    text = re.sub(r'#(\w+)', r'\1', text)

    # 4. Remove diacritics (fatha, damma, kasra, tanwin, shadda, sukun...)
    #    Unicode range U+064B–U+065F covers all Arabic diacritics
    text = re.sub(r'[\u064B-\u065F]', '', text)

    # 5. Normalize alef variants  (أ إ آ ٱ → ا)
    text = re.sub(r'[أإآٱ]', 'ا', text)

    # 6. Normalize teh marbuta (ة → ه)
    #    These are the same sound; treating them as different tokens is noise
    text = re.sub(r'ة', 'ه', text)

    # 7. Normalize alef maqsura (ى → ي)
    text = re.sub(r'ى', 'ي', text)

    # 8. Remove tatweel / kashida (elongation character ـ)
    text = re.sub(r'ـ', '', text)

    # 9. Remove non-Arabic, non-space characters
    #    Keeps Arabic letters (U+0600–U+06FF), spaces, and basic punctuation
    text = re.sub(r'[^\u0600-\u06FF\s]', '', text)

    # 10. Collapse multiple spaces into one
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# Quick test to see it working
examples = [
    "مَرْحَبًا بِكَ!!! http://example.com @someone #تونس",   # has diacritics, url, mention, hashtag
    "الطَّالِبَةُ مُجْتَهِدَةٌ جِدًّا",                      # heavy diacritics
    "جمييييييل جداً",                                         # tatweel / elongation
    "هاذه الأمة ستنتصر إن شاء الله",                          # alef variants
]

for e in examples:
    print(f"IN:  {e}")
    print(f"OUT: {clean_arabic(e)}")
    print()

## Step 3 — Apply to the whole dataset, compare before/after

Always look at before/after pairs on real data — not just invented test cases. You'll often find edge cases you didn't think of.

In [ ]:
df = df_raw.copy()
df['text_clean'] = df['text'].apply(clean_arabic)

# Side-by-side: raw vs cleaned on 8 random rows
sample = df.sample(8, random_state=1)[['text', 'text_clean', 'sentiment']]
pd.set_option('display.max_colwidth', 120)
sample

In [ ]:
# Check if cleaning produced any empty strings
empty_after_clean = df[df['text_clean'].str.strip() == '']
print(f'Rows that became empty after cleaning: {len(empty_after_clean)}')

if len(empty_after_clean) > 0:
    print('\nThese rows:')
    display(empty_after_clean[['text', 'sentiment']])

In [ ]:
# How much did cleaning reduce tweet length?
df['char_count_raw'] = df['text'].str.len()
df['char_count_clean'] = df['text_clean'].str.len()
df['chars_removed'] = df['char_count_raw'] - df['char_count_clean']

print('Characters removed per tweet:')
print(df['chars_removed'].describe().round(1))
print()
print(f'Total characters removed: {df["chars_removed"].sum():,}')
print(f'Avg reduction per tweet: {df["chars_removed"].mean():.1f} chars ({df["chars_removed"].mean() / df["char_count_raw"].mean() * 100:.1f}%)')

## Step 4 — Drop empty rows and save

Drop any rows where cleaning wiped out the entire text, then save to `data/processed/`.

The processed file is what every notebook from here on will load — not the raw file.

In [ ]:
df_clean = (
    df[['text_clean', 'sentiment', 'label']]
    .rename(columns={'text_clean': 'text'})
    .loc[lambda x: x['text'].str.strip() != '']
    .reset_index(drop=True)
)

print(f'Before: {len(df)} rows')
print(f'After:  {len(df_clean)} rows  ({len(df) - len(df_clean)} dropped)')
print()
print('Final class distribution:')
print(df_clean['sentiment'].value_counts())
print()
df_clean.head(5)

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

output_path = '../data/processed/tweets_clean.csv'
df_clean.to_csv(output_path, index=False)

print(f'Saved to {output_path}')
print(f'Columns: {df_clean.columns.tolist()}')

## Findings

*(Fill in after running)*

Things to note:
- Did any rows become empty after cleaning? If yes, what were they?
- What percentage of characters were removed on average?
- Did you spot any cleaning decisions you'd do differently? (e.g. was removing ة → ه the right call?)
- Is the 864-char outlier still in the dataset, or did cleaning handle it?